# Assignment 2 (Week 5): Leak-free cleaning of a noisy rover log
# تکلیف ۲ (هفته‌ی ۵): پاک‌سازی بدون نشت داده‌ی یک لاگ نویزی ربات

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 1 &nbsp;|&nbsp;
**Released:** Week 5 &nbsp;|&nbsp; **Due:** start of Week 6 &nbsp;|&nbsp; **Effort:** about 4 hours &nbsp;|&nbsp;
**Weight:** graded milestone A2 (see the syllabus)

> **Provenance:** new content. Missing-data mechanisms after Little and Rubin, *Statistical
> Analysis with Missing Data* (2019), Ch. 1; Tukey's fences from Tukey, *Exploratory Data
> Analysis* (1977), Ch. 2; cleaning and leakage rules rewritten (not copied) from W3Schools
> ("Pandas - Cleaning Data") and GeeksforGeeks ("Handling Missing Values in Machine Learning",
> "Detect and Remove the Outliers using Python", "Data Leakage").

<div dir="rtl">

## صورت مسئله

ربات خودران پروژه‌ی نهایی (ماژول M3) برای تصمیم ترمز به **فاصله‌ی مانع در یک ثانیه‌ی بعد** نیاز دارد. فایل <code>data/sample/assignment02_rover_log.csv</code> شانزده آزمون رانندگی دیگر (بذر تصادفی متفاوت با کلاس) را با همان نقص‌ها دارد: صفرهای LiDAR («بازتابی دریافت نشد»)، شکاف‌های NaN در سرعت چرخ، جهش‌های شتاب‌سنج، پژواک گردوغبار، ژیروسکوپ یخ‌زده و سطرهای تکراری. کار شما ساختن مسیر کامل از فایل خام تا آرایه‌های آماده‌ی مدل است، **بدون هیچ نشت داده‌ای**:

۱. **ممیزی:** تابع <code>audit_report</code> تعداد سطرهای تکراری، صفرهای LiDAR «ابتدایی» (پیش از اولین خوانش معتبر هر اجرا؛ دیوار دورتر از ۱۲ متر، یعنی MNAR) و «پراکنده» (افت تصادفی، MCAR) و NaNهای سرعت چرخ را بشمارد.

۲. **حسگر یخ‌زده:** مرحله‌ی <code>FlagStuck</code> را بنویسید که هر رشته‌ی دست‌کم <code>min_run</code> مقدار کاملاً یکسانِ پشت‌سرهم را به NaN تبدیل کند.

۳. **حصارهای توکی:** مرحله‌ی آموختنی <code>IQRClipper</code> را (زیرکلاس <code>ColumnStatStage</code>) بنویسید که حصارها را فقط از داده‌ی آموزش یاد بگیرد و مقادیر بیرون را به حصار ببرد.

۴. **دستورالعمل‌ها:** <code>build_cleaning_pipeline</code> (فقط مراحل بدون حالت) و <code>build_learned_pipeline</code> (مراحل آموختنی) را بسازید.

۵. **آماده‌سازی بدون نشت:** تابع <code>prepare</code> فایل خام را پاک کند، اجراها را (نه سطرها را) به آموزش و آزمون تقسیم کند، مراحل آموختنی را فقط روی آموزش برازش دهد و گزارش نشت را برگرداند.

۶. **آزمون و گزارش:** دست‌کم پنج آزمون دیگر در <code>test_sensor_cleaning.py</code> بنویسید و به سه پرسش انتهای این دفترچه کوتاه پاسخ دهید.

کد را در فایل <code>sensor_cleaning.py</code> بنویسید؛ این دفترچه فقط ماژول شما را بارگذاری و بررسی می‌کند. به هیچ فایلی در <code>src/</code> دست نزنید.

</div>

**In English.** Build the whole path from the raw CSV to model-ready arrays without leakage:
count the defects (Task 1), detect a frozen sensor (Task 2), write a learned Tukey-fence clipper
(Task 3), assemble the stateless and the learned pipelines (Task 4), split by run and fit the
learned stages on the training runs only (Task 5), and test your code and answer three short
questions (Task 6). Write your code in `sensor_cleaning.py`; this notebook only loads and checks it.

## Setup (run first, do not edit)

In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path
from types import ModuleType
from typing import Final

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
ASSIGNMENT_DIR: Final[Path] = REPO_ROOT / "assignments" / "assignment02"
for folder in (REPO_ROOT / "src", ASSIGNMENT_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from aerospace_ai.models import LinearLeastSquares  # noqa: E402
from aerospace_ai.preprocessing import PerGroup, audit_frame  # noqa: E402

RAW_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "assignment02_rover_log.csv"
raw = pd.read_csv(RAW_CSV)


def load_student_module() -> ModuleType:
    """(Re)import sensor_cleaning.py so edits are picked up without a kernel restart."""
    if "sensor_cleaning" in sys.modules:
        return importlib.reload(sys.modules["sensor_cleaning"])
    return importlib.import_module("sensor_cleaning")


def status(task: str, check: object) -> None:
    """Run a public check and print a short status line."""
    try:
        check()
        print(f"[x] {task}: public checks passed")
    except NotImplementedError:
        print(f"[ ] {task}: not done yet")
    except AssertionError as err:
        print(f"[!] {task}: check failed: {err}")


print("Repository root found:", REPO_ROOT.name, "| raw rows:", len(raw))
display(audit_frame(raw, {"lidar_range_m": 0.0}).round(3))

### Task 1 (15 pts): `audit_report`
<div dir="rtl">سطرهای تکراری را روی جدول خام بشمارید؛ بقیه را پس از مرتب‌سازی و حذف تکرارها. صفر LiDAR «ابتدایی» است اگر پیش از اولین خوانش غیرصفرِ همان اجرا بیاید.</div>

By hand first: run 1 reads `0, 0, 5.0, 0` and run 2 reads `4.0, 0, 3.9`. Leading zeros: 2 (both
at the start of run 1); isolated zeros: 2 (one in each run).

In [ ]:
def _check_task1() -> None:
    sc = load_student_module()
    counts = sc.audit_report(raw)
    print(counts, "| all LiDAR zeros:", counts.lidar_zeros)
    assert counts.lidar_zeros == int((raw.drop_duplicates(["run_id", "time_s"])["lidar_range_m"] == 0).sum())
    assert counts.lidar_leading_zeros > counts.lidar_isolated_zeros, "most zeros are the wall beyond range"


status("Task 1", _check_task1)

In [ ]:
# HIDDEN TESTS (task1): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 2 (20 pts): `FlagStuck`
<div dir="rtl">رشته‌های مقادیر برابر را با <code>np.cumsum</code> برچسب بزنید، طول هر رشته را با <code>np.bincount</code> بیابید و با <code>min_run</code> مقایسه کنید. NaN هرگز رشته‌ی یخ‌زده نیست.</div>

By hand first: `min_run=3`, values `[1, 2, 2, 2, 3, 3]` → mask `[F, T, T, T, F, F]`.

In [ ]:
def _check_task2() -> None:
    sc = load_student_module()
    log = raw.drop_duplicates(["run_id", "time_s"]).reset_index(drop=True)
    out = PerGroup(sc.FlagStuck(("gyro_z_dps",)), "run_id")(log)
    flagged = out["gyro_z_dps"].isna()
    print("frozen gyro samples:", int(flagged.sum()), "in runs", sorted(out.loc[flagged, "run_id"].unique()))
    assert 0 < flagged.sum() < 100


status("Task 2", _check_task2)

In [ ]:
# HIDDEN TESTS (task2): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 3 (15 pts): `IQRClipper`
<div dir="rtl">چارک‌ها را از داده‌ی آموزش بگیرید و حصارهای Q1 − k·IQR و Q3 + k·IQR را ذخیره کنید؛ در اعمال، مقادیر را با <code>Series.clip</code> به حصارها محدود کنید.</div>

By hand first: training column `[1, 2, 3, 4, 100]` has $Q_1 = 2$, $Q_3 = 4$ (pandas' linear
interpolation), IQR = 2, so the $k = 1.5$ fences are $[-1, 7]$ and the $k = 3$ fences $[-4, 10]$.

In [ ]:
def _check_task3() -> None:
    sc = load_student_module()
    clipper = sc.IQRClipper(("a",)).fit(pd.DataFrame({"a": [1.0, 2.0, 3.0, 4.0, 100.0]}))
    print(clipper, clipper.stats_)
    out = clipper(pd.DataFrame({"a": [-5.0, 0.0, 50.0]}))
    assert out["a"].tolist() == [-1.0, 0.0, 7.0]


status("Task 3", _check_task3)

In [ ]:
# HIDDEN TESTS (task3): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 4 (15 pts): the two recipes
<div dir="rtl">دستورالعمل بدون حالت را از دستورالعمل کلاس بسازید و <code>FlagStuck</code> را (برای هر اجرا جداگانه) درست پس از مرحله‌ی اول قرار دهید. دستورالعمل آموختنی سه مرحله دارد.</div>

In [ ]:
def _check_task4() -> None:
    sc = load_student_module()
    cleaning, learned = sc.build_cleaning_pipeline(), sc.build_learned_pipeline()
    print(cleaning, "\n", learned)
    assert not any(stage.REQUIRES_FIT for stage in cleaning), "learned stages do not belong here"
    assert all(stage.REQUIRES_FIT for stage in learned)
    clean = cleaning.fit_transform(raw)
    print("rows after cleaning:", len(clean), "| gyro NaN:", int(clean["gyro_z_dps"].isna().sum()))


status("Task 4", _check_task4)

In [ ]:
# HIDDEN TESTS (task4): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 5 (25 pts): `prepare`, leak-free
<div dir="rtl">اجراهای ۱ تا ۱۲ آموزش و ۱۳ تا ۱۶ آزمون‌اند. هیچ آماری (میانه، چارک) نباید از سطرهای آزمون آمده باشد.</div>

In [ ]:
def _check_task5() -> None:
    sc = load_student_module()
    data = sc.prepare(RAW_CSV)
    print("train", data.X_train.shape, "runs", data.train_runs)
    print("test ", data.X_test.shape, "runs", data.test_runs)
    print(data.leakage)
    model = LinearLeastSquares().fit(data.X_train, data.y_train)
    rmse = model.score(data.X_test, data.y_test)
    print(f"linear test RMSE on new drives: {rmse:.4f} m")
    assert data.leakage.ok, "the split leaks"
    assert rmse < 0.2, "far worse than expected: check the cleaning recipe"


status("Task 5", _check_task5)

In [ ]:
# HIDDEN TESTS (task5): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 6 (10 pts): your tests and three short answers
<div dir="rtl">دست‌کم پنج آزمون دیگر در <code>test_sensor_cleaning.py</code> بنویسید (fixture، <code>parametrize</code> و <code>pytest.raises</code>). سپس در یک سلول Markdown، به فارسی یا انگلیسی و هر کدام در دو یا سه جمله، پاسخ دهید:</div>

1. Why are the leading LiDAR zeros dropped instead of imputed? Name the mechanism.
2. With `k = 1.5` instead of `k = 3` the test RMSE gets slightly *worse*. What physical signal do
   the narrow fences clip?
3. A classmate splits the cleaned rows at random and reports a k-NN RMSE about half of yours.
   Who is right, and what does `check_split` say about their split?

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "test_sensor_cleaning.py"],
    cwd=ASSIGNMENT_DIR, capture_output=True, text=True,
)
print(result.stdout[-2000:])

In [ ]:
# HIDDEN TESTS (task6): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

*Your three answers here (Markdown).*

## Expected output (check yourself against these)

| Quantity | Expected value |
|----------|----------------|
| Task 1 `DefectCounts` | duplicate_rows 32, lidar_leading_zeros 482, lidar_isolated_zeros 50, speed_nan 25 |
| Task 2 frozen gyro samples | 62 (two segments of 31, runs 3 and 15) |
| Task 3 hand example | fences (−1, 7) for k = 1.5, (−4, 10) for k = 3 |
| Task 4 rows after cleaning | 1919, of which 62 have a NaN gyro value |
| Task 5 shapes | X_train (1426, 4), X_test (493, 4); runs 1–12 vs 13–16; leakage report OK |
| Task 5 linear test RMSE | 0.109 m (0.112 m with k = 1.5) |
| Task 6 | at least 7 tests, all passing; three answers |